# Lesson 2: RAG Triad of metrics

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import utils

print(f"LLM/embeddings via Gemini + local HF | API key loaded: {bool(utils.get_google_api_key())}")


In [ ]:
import nest_asyncio

nest_asyncio.apply()


In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["./eBook-How-to-Build-a-Career-in-AI.pdf"]
).load_data()


In [ ]:
from llama_index.core import Document

document = Document(text="\n\n".join([doc.text for doc in documents]))


In [ ]:
from utils import build_sentence_window_index
from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(
    model="gemini-2.5-flash",
    api_key=utils.get_google_api_key(),
    temperature=0.1,
)

sentence_index = build_sentence_window_index(
    document,
    llm,
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="sentence_index"
)

In [ ]:
from utils import get_sentence_window_query_engine

sentence_window_engine = \
get_sentence_window_query_engine(sentence_index)

In [ ]:
output = sentence_window_engine.query(
    "How do you create your AI portfolio?")
output.response

## RAG triad evaluators (LlamaIndex + Gemini)

We score each answer with three LlamaIndex evaluators (judge = Gemini, temperature=0):

1. **Answer Relevancy** — does the answer address the question?
2. **Context Relevancy** — is retrieved context useful for the question?
3. **Groundedness (Faithfulness)** — is the answer supported by the retrieved context?

Scores are in the range 0–1; higher is better.


In [ ]:
# nest_asyncio already applied above
pass


In [ ]:
from utils import get_eval_llm, get_rag_triad_evaluators

eval_llm = get_eval_llm()  # Gemini judge, temperature=0
evaluators = get_rag_triad_evaluators(llm=eval_llm)
evaluators


### 1. Answer Relevancy


In [ ]:
answer_relevancy = evaluators["answer_relevancy"]
answer_relevancy


### 2. Context Relevancy


In [ ]:
context_relevancy = evaluators["context_relevancy"]
context_relevancy


In [ ]:
# Demo: score one answer + its retrieved contexts
sample_q = "How do you create your AI portfolio?"
sample_response = sentence_window_engine.query(sample_q)
sample_contexts = [n.node.get_content() for n in sample_response.source_nodes]

ar = answer_relevancy.evaluate(query=sample_q, response=str(sample_response))
cr = context_relevancy.evaluate(query=sample_q, contexts=sample_contexts)
print("Answer relevancy:", ar.score, ar.feedback)
print("Context relevancy:", cr.score, cr.feedback)


In [ ]:
pass  # context relevancy demo is in the previous cell


### 3. Groundedness (Faithfulness)


In [ ]:
groundedness = evaluators["groundedness"]
groundedness


In [ ]:
fa = groundedness.evaluate_response(response=sample_response)
print("Groundedness:", fa.score, getattr(fa, "feedback", None) or fa.passing)


## Evaluation of the RAG application


In [ ]:
from utils import evaluate_query_engine, summarize_eval_df


In [ ]:
eval_questions = []
with open("eval_questions.txt", "r") as file:
    for line in file:
        item = line.strip()
        if item:
            eval_questions.append(item)


In [ ]:
eval_questions


In [ ]:
eval_questions.append("How can I be successful in AI?")


In [ ]:
eval_questions


In [ ]:
records = evaluate_query_engine(
    sentence_window_engine,
    eval_questions,
    app_id="sentence_window",
    llm=eval_llm,
    include_feedback=True,
)
records.head()


In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)
records[
    [
        "question",
        "answer_relevancy",
        "context_relevancy",
        "groundedness",
    ]
]


In [ ]:
# Optional: inspect judge feedback for the first few rows
records[
    [
        "question",
        "answer_relevancy_feedback",
        "context_relevancy_feedback",
        "groundedness_feedback",
    ]
].head()


In [ ]:
summarize_eval_df(records)


In [ ]:
print("Done. Use summarize_eval_df(records) for mean triad scores.")
